# Databricks ↔ Snowflake Iceberg Interoperability

## Overview
This notebook demonstrates **bidirectional** Iceberg interoperability between Snowflake and Azure Databricks using:
- **Snowflake → Databricks**: Catalog-Linked Database (CLD) with Unity Catalog REST API
- **Databricks → Snowflake**: Databricks reading Snowflake Iceberg tables via IRC

## Architecture
```
┌─────────────────────────────────────────────────────────────────┐
│                    BIDIRECTIONAL INTEROP                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ┌──────────────┐         REST API          ┌───────────────┐  │
│  │  Snowflake   │ ◄─────────────────────────►│  Databricks   │  │
│  │              │   Unity Catalog / IRC      │ Unity Catalog │  │
│  │  CLD Query   │                            │  UniForm      │  │
│  └──────┬───────┘                            └───────┬───────┘  │
│         │                                            │          │
│         │  Vended Credentials                        │          │
│         ▼                                            ▼          │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │            Azure Blob Storage (ADLS Gen2)               │   │
│  │   abfss://gf-unity@gfstorageaccountwest2.dfs.core...    │   │
│  │                                                         │   │
│  │   Delta Tables + UniForm (Iceberg metadata)             │   │
│  └─────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────┘
```

## Test Cases
| Direction | Method | Tables | Target |
|-----------|--------|--------|--------|
| SF → DBX | CLD + Unity Catalog IRC | 5 healthcare tables | Query latency <5s |
| DBX → SF | Spark IRC Catalog | Iceberg POC tables | Full schema access |

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;

---
# Part 1: Snowflake Reading from Databricks (CLD)

## Configuration Details
| Component | Value |
|-----------|-------|
| Databricks Workspace | `https://adb-2584487012733217.17.azuredatabricks.net` |
| Unity Catalog | `gf_interop` |
| Schema | `uniform` (Delta + UniForm enabled) |
| Storage | `abfss://gf-unity@gfstorageaccountwest2.dfs.core.windows.net/` |
| Catalog Integration | `gf_interop_unity_int` |
| CLD Database | `gf_interop_cld` |

## Step 1.1: Verify Catalog Integration
The catalog integration connects Snowflake to Databricks Unity Catalog via the Iceberg REST API.

In [ ]:
-- Show existing catalog integration
SHOW CATALOG INTEGRATIONS LIKE 'gf_interop%';

-- Describe the integration details
DESCRIBE CATALOG INTEGRATION gf_interop_unity_int;

## Step 1.2: Verify Catalog-Linked Database (CLD)
The CLD automatically discovers and syncs tables from Databricks Unity Catalog.

In [ ]:
-- Show the CLD database
SHOW DATABASES LIKE 'gf_interop_cld';

-- Describe database properties
DESCRIBE DATABASE gf_interop_cld;

-- List schemas in CLD
SHOW SCHEMAS IN DATABASE gf_interop_cld;

In [ ]:
-- List all tables auto-discovered from Databricks
SHOW ICEBERG TABLES IN SCHEMA gf_interop_cld.uniform;

-- Alternative: Show all tables
SHOW TABLES IN SCHEMA gf_interop_cld.uniform;

## Step 1.3: Query Databricks Tables from Snowflake
These tables are Delta tables in Databricks with UniForm enabled, exposed as Iceberg to Snowflake.

In [ ]:
-- Query PATIENTS table
SELECT 'PATIENTS' AS table_name, COUNT(*) AS row_count FROM gf_interop_cld.uniform.patients
UNION ALL
SELECT 'CLAIMS', COUNT(*) FROM gf_interop_cld.uniform.claims
UNION ALL
SELECT 'ENCOUNTERS', COUNT(*) FROM gf_interop_cld.uniform.encounters
UNION ALL
SELECT 'MEDICATIONS', COUNT(*) FROM gf_interop_cld.uniform.medications
UNION ALL
SELECT 'PROVIDERS', COUNT(*) FROM gf_interop_cld.uniform.providers;

In [ ]:
-- Sample data from patients table
SELECT * FROM gf_interop_cld.uniform.patients LIMIT 10;

In [ ]:
-- Describe table schema
DESCRIBE TABLE gf_interop_cld.uniform.patients;

## Step 1.4: Cross-Platform Analytics
Join Databricks-managed tables with Snowflake-native tables.

In [ ]:
-- Join Databricks claims with patient data for analytics
SELECT 
    p.Id AS patient_id,
    p.FIRST,
    p.LAST,
    p.BIRTHDATE,
    COUNT(c.Id) AS claim_count,
    SUM(c.TOTAL_CLAIM_COST) AS total_claims
FROM gf_interop_cld.uniform.patients p
LEFT JOIN gf_interop_cld.uniform.claims c ON p.Id = c.PATIENTID
GROUP BY 1, 2, 3, 4
ORDER BY total_claims DESC NULLS LAST
LIMIT 20;

In [ ]:
-- Healthcare utilization analysis
SELECT 
    e.ENCOUNTERCLASS,
    COUNT(DISTINCT e.PATIENT) AS unique_patients,
    COUNT(*) AS encounter_count,
    AVG(e.TOTAL_CLAIM_COST) AS avg_cost
FROM gf_interop_cld.uniform.encounters e
GROUP BY 1
ORDER BY encounter_count DESC;

In [ ]:
-- Medication analysis
SELECT 
    DESCRIPTION,
    COUNT(*) AS prescription_count,
    AVG(TOTALCOST) AS avg_cost
FROM gf_interop_cld.uniform.medications
GROUP BY 1
ORDER BY prescription_count DESC
LIMIT 15;

## Step 1.5: Performance Benchmark - CLD Queries

In [ ]:
-- Benchmark: Full table scan
SELECT 
    'Full Scan' AS test,
    COUNT(*) AS rows,
    CURRENT_TIMESTAMP() AS executed_at
FROM gf_interop_cld.uniform.claims;

In [ ]:
-- Benchmark: Aggregation query
SELECT 
    DATE_TRUNC('month', TO_DATE(BILLABLEPERIOD_START)) AS month,
    COUNT(*) AS claim_count,
    SUM(TOTAL_CLAIM_COST) AS total_cost,
    AVG(TOTAL_CLAIM_COST) AS avg_cost
FROM gf_interop_cld.uniform.claims
WHERE BILLABLEPERIOD_START IS NOT NULL
GROUP BY 1
ORDER BY 1;

In [ ]:
-- Check query performance from history
SELECT 
    QUERY_TEXT,
    TOTAL_ELAPSED_TIME / 1000 AS elapsed_sec,
    BYTES_SCANNED / 1024 / 1024 AS mb_scanned,
    ROWS_PRODUCED
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY())
WHERE QUERY_TEXT ILIKE '%gf_interop_cld%'
    AND QUERY_TYPE = 'SELECT'
ORDER BY START_TIME DESC
LIMIT 10;

---
# Part 2: Databricks Reading from Snowflake (IRC)

This section documents how to configure Databricks to read Snowflake Iceberg tables.

## Snowflake IRC Endpoint
```
https://sfsenorthamerica-demo_gfuribondo2.snowflakecomputing.com/polaris/api/catalog
```

## Step 2.1: Databricks Cluster Configuration

Add these Spark configurations to your Databricks cluster:

```
spark.sql.catalog.snowflake_catalog org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.snowflake_catalog.type rest
spark.sql.catalog.snowflake_catalog.uri https://sfsenorthamerica-demo_gfuribondo2.snowflakecomputing.com/polaris/api/catalog
spark.sql.catalog.snowflake_catalog.credential <SNOWFLAKE_PAT>
spark.sql.catalog.snowflake_catalog.warehouse ICEBERG_POC
spark.sql.catalog.snowflake_catalog.scope session:role:ACCOUNTADMIN
```

## Step 2.2: Sample Databricks PySpark Code

```python
# Run this in a Databricks notebook
from pyspark.sql import SparkSession

# List namespaces from Snowflake
spark.sql("SHOW NAMESPACES IN snowflake_catalog").show()

# List tables
spark.sql("SHOW TABLES IN snowflake_catalog.ICEBERG_POC.TESTS").show()

# Read Snowflake Iceberg table
df = spark.read.table("snowflake_catalog.ICEBERG_POC.TESTS.EVENTS")
df.show(10)

# Run analytics
df.groupBy("event_type", "region").count().show()
```

---
# Part 3: CLD Setup Reference (For Recreation)

If you need to recreate the CLD setup, here are the steps:

In [ ]:
-- REFERENCE: Create Catalog Integration for Unity Catalog (already exists)
-- NOTE: Uses Iceberg REST API endpoint with vended credentials

/*
CREATE OR REPLACE CATALOG INTEGRATION gf_interop_unity_int
  CATALOG_SOURCE = ICEBERG_REST
  TABLE_FORMAT = ICEBERG
  CATALOG_NAMESPACE = 'gf_interop.uniform'
  REST_CONFIG = (
    CATALOG_URI = 'https://adb-2584487012733217.17.azuredatabricks.net/api/2.1/unity-catalog/iceberg-rest'
    WAREHOUSE = 'gf_interop'
    ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
  )
  REST_AUTHENTICATION = (
    TYPE = BEARER
    BEARER_TOKEN = '<DATABRICKS_PAT>'
  )
  ENABLED = TRUE;
*/

In [ ]:
-- REFERENCE: Create Catalog-Linked Database (already exists)

/*
CREATE OR REPLACE DATABASE gf_interop_cld
  FROM LINKED_CATALOG
  CATALOG_INTEGRATION = 'gf_interop_unity_int';
*/

## Databricks Prerequisites for Vended Credentials

In Databricks, you must grant `EXTERNAL USE SCHEMA` to allow credential vending:

```sql
-- Run in Databricks SQL
GRANT EXTERNAL USE SCHEMA ON SCHEMA gf_interop.uniform TO `user@domain.com`;

-- Enable UniForm on Delta tables
ALTER TABLE gf_interop.uniform.patients 
  SET TBLPROPERTIES ('delta.universalFormat.enabledFormats' = 'iceberg');

-- Run OPTIMIZE to generate Iceberg metadata
OPTIMIZE gf_interop.uniform.patients;
```

---
# Part 4: Interoperability Validation

In [ ]:
-- Validate all CLD tables are accessible
SELECT 
    'gf_interop_cld.uniform.patients' AS table_fqn,
    (SELECT COUNT(*) FROM gf_interop_cld.uniform.patients) AS row_count,
    'PASS' AS status
UNION ALL
SELECT 'gf_interop_cld.uniform.claims', (SELECT COUNT(*) FROM gf_interop_cld.uniform.claims), 'PASS'
UNION ALL
SELECT 'gf_interop_cld.uniform.encounters', (SELECT COUNT(*) FROM gf_interop_cld.uniform.encounters), 'PASS'
UNION ALL
SELECT 'gf_interop_cld.uniform.medications', (SELECT COUNT(*) FROM gf_interop_cld.uniform.medications), 'PASS'
UNION ALL
SELECT 'gf_interop_cld.uniform.providers', (SELECT COUNT(*) FROM gf_interop_cld.uniform.providers), 'PASS';

In [ ]:
-- Get Iceberg table metadata for a CLD table
SELECT SYSTEM$GET_ICEBERG_TABLE_INFORMATION('gf_interop_cld.uniform.patients') AS iceberg_info;

---
# Summary

## What Was Tested
| Test | Result | Notes |
|------|--------|-------|
| Catalog Integration | ✅ PASS | Unity Catalog REST API connected |
| CLD Auto-Discovery | ✅ PASS | 5 tables synced automatically |
| Vended Credentials | ✅ PASS | EXTERNAL USE SCHEMA required |
| Query Performance | ✅ PASS | Sub-5s latency on all queries |
| Cross-Platform Joins | ✅ PASS | Databricks + Snowflake data joined |

## Key Findings
1. **UniForm is required** - Delta tables must have UniForm enabled to expose Iceberg metadata
2. **EXTERNAL USE SCHEMA** - Must be granted in Databricks for credential vending
3. **OPTIMIZE required** - Delta tables need OPTIMIZE to generate initial Iceberg metadata
4. **Use iceberg-rest endpoint** - Legacy `/unity-catalog/iceberg` endpoint is deprecated

## Connection Details
| Component | Value |
|-----------|-------|
| Snowflake Account | SFSENORTHAMERICA-DEMO_GFURIBONDO2 |
| Databricks Workspace | https://adb-2584487012733217.17.azuredatabricks.net |
| Unity Catalog | gf_interop |
| Schema | uniform |
| Tables | patients, claims, encounters, medications, providers |